## Complete the semantic alignment of ICD codes

### Step 1: Convert ICD-9 to ICD-10

**Use the mapping provided by the ETHOS model to convert ICD-9 codes to ICD-10 codes.**

In [1]:
import pandas as pd

# File paths
vocab_file = '../vocab_stoi.csv'
mapping_file = 'icd_cm_9_to_10_mapping_from_ethos.csv'

# Step 1: Load Data
print("Loading files...")
vocab_df = pd.read_csv(vocab_file)  # ICD-9 vocabulary
mapping_df = pd.read_csv(mapping_file)  # ICD-9 to ICD-10 mapping

# Step 2: Convert Mapping to Dictionary for Fast Lookup
icd9_to_icd10 = dict(zip(mapping_df['icd_9'], mapping_df['icd_10']))

# Step 3: Map ICD-9 Codes to ICD-10
print("Mapping ICD-9 codes to ICD-10...")
mapped_results = []
unmapped_icd9 = []

for _, row in vocab_df.iterrows():
    icd9_code = row['String']  # Extract ICD-9 code
    token_index = row['Token Index']  # Extract token index
    icd10_code = icd9_to_icd10.get(icd9_code, None)  # Map to ICD-10

    if icd10_code:  # If mapping exists
        mapped_results.append({'ICD-9': icd9_code, 'ICD-10': icd10_code, 'Token Index': token_index})
    else:  # If no mapping exists
        unmapped_icd9.append({'ICD-9': icd9_code, 'Token Index': token_index})
        mapped_results.append({'ICD-9': icd9_code, 'ICD-10': 'UNKNOWN', 'Token Index': token_index})

# Step 5: Summary and Statistics
unmapped_count = len(unmapped_icd9)
total_count = len(vocab_df)

print(f"Total ICD-9 codes: {total_count}")


Loading files...
Mapping ICD-9 codes to ICD-10...
Total ICD-9 codes: 6984


### Step 2: Validate ICD-10 Codes Against Hierarchical Graph

<!-- ### Step 2: Validate ICD-10 Codes Against Hierarchical Graph -->

In [2]:
mapped_results_df = pd.DataFrame(mapped_results)
assert mapped_results_df['Token Index'].unique().shape[0] == total_count
mapped_results_df.to_csv('vocab_stoi_icd10.csv', index=False)

In [3]:
import json
print("Loading ICD10 hierarchy file...")
with open('icd10_hierarchical_graph.json', 'r') as file:
    icd10_hierarchical_graph = json.load(file)


Loading ICD10 hierarchy file...


In [4]:
def normalize_icd10_code(icd10_code, icd10_hierarchy):
    """Normalize ICD-10 code by truncating or finding the nearest match."""
    if icd10_code in icd10_hierarchy:
        return icd10_code  # Code already exists
    if '.' not in icd10_code:
        icd10_code = icd10_code[:3] + '.' + icd10_code[3:]
    # Truncate trailing characters (e.g., "S065X9A" -> "S06.5X")
    flag = False
    search_code = None
    for length in range(3, len(icd10_code)+1):
        if length == 4:
            # Skip the fourth character, e.g., 'S06.'
            continue
        truncated_code = icd10_code[:length]
        if truncated_code in icd10_hierarchy:
            search_code = truncated_code
            flag = True
        else:
            if flag:
                break
    # Return None if no match is found
    return search_code

mapped_results_df['ICD-10-Align'] = mapped_results_df['ICD-10'].apply(lambda x: normalize_icd10_code(x, icd10_hierarchical_graph))
mapped_results_df = mapped_results_df[['Token Index', 'ICD-9', 'ICD-10', 'ICD-10-Align']]
mapped_results_df.to_csv('vocab_stoi_icd10.csv', index=False)
print(f'{mapped_results_df['ICD-10-Align'].isna().sum()} codes need to be extra checked.')

# Sort the DataFrame, placing rows with NaN in 'ICD-10-Align' at the bottom
sort_mapped_results_df = mapped_results_df.sort_values(
    by='ICD-10-Align',
    key=lambda col: col.isna(),
    ascending=True
)
sort_mapped_results_df.to_csv('vocab_stoi_icd10.csv', index=False)

358 codes need to be extra checked.


**Note, extra operations**
1. there're almost over 40 ICD codes converted into Unknown
2. Some converted ICD-10 codes are not following instructions, i.e., '2898,ICD-9 code 289.8 maps to ICD-10 code D75.89.,,5446'
3. Some ICD-9 codes are mapped to 'NoDx' using `icd_cm_9_to_10_mapping.csv`

For these cases, we need to use ChatGPT to conver them again until each code is mapped to a valid ICD-10 code that can be found in the hierarchical graph. (For use of ChatGPT due to inavailability of web search in API calling)

After conversion, we save the mapping for these 358 codes in to a supplementary file `icd_cm_9_to_10_mapping_supplementary.csv`.


In [5]:
# Load supplementary mapping data
supplementary_mapping_df = pd.read_csv('icd_cm_9_to_10_mapping_supplementary.csv')

# Iterate over rows of the mapped results DataFrame
for index, row in sort_mapped_results_df.iterrows():
    if pd.isna(row['ICD-10-Align']):  # Check if ICD-10-Align is NaN
        # Find corresponding ICD-10 code for the ICD-9 code
        supp_icd_10_code = supplementary_mapping_df[supplementary_mapping_df['icd_9'] == row['ICD-9']]['icd_10'].values
        # Check if corresponding ICD-10 code was found
        if len(supp_icd_10_code) == 0:
            raise ValueError(f'No ICD-10 code found for ICD-9 code: {row["ICD-9"]}')
        
        # Assign the found ICD-10 code to the row
        sort_mapped_results_df.loc[index, 'ICD-10'] = supp_icd_10_code[0]


In [6]:
del sort_mapped_results_df['ICD-10-Align']
sort_mapped_results_df

,Token Index,ICD-9,ICD-10
0,0,78002,R404
4600,4600,59370,N1371
4599,4599,37000,H16009
4598,4598,8439,S76919A
4597,4597,8448,S86819A
...,...,...,...
3732,3732,E9051,W57.XXXA
5767,5767,E9430,T43.0X3A
6620,6620,6009,N40.3
3093,3093,E8716,Y65.8


In [7]:
sort_mapped_results_df['ICD-10-Align-Hierarchy'] = sort_mapped_results_df['ICD-10'].apply(lambda x: normalize_icd10_code(x, icd10_hierarchical_graph))

In [8]:
sort_mapped_results_df

,Token Index,ICD-9,ICD-10,ICD-10-Align-Hierarchy
0,0,78002,R404,R40.4
4600,4600,59370,N1371,N13.71
4599,4599,37000,H16009,H16.00
4598,4598,8439,S76919A,S76.91
4597,4597,8448,S86819A,S86.81
...,...,...,...,...
3732,3732,E9051,W57.XXXA,W57
5767,5767,E9430,T43.0X3A,T43.0
6620,6620,6009,N40.3,N40.3
3093,3093,E8716,Y65.8,Y65.8


In [9]:
# sort_mapped_results_df = sort_mapped_results_df.rename(columns={'ICD-10': 'ICD-10-Align-Hierarchy'})
final_mapped_results_df = sort_mapped_results_df
final_mapped_results_df.sort_values(by='Token Index', inplace=True)
final_mapped_results_df = final_mapped_results_df[['Token Index', 'ICD-9', 'ICD-10', 'ICD-10-Align-Hierarchy']]

In [10]:
final_mapped_results_df

,Token Index,ICD-9,ICD-10,ICD-10-Align-Hierarchy
0,0,78002,R404,R40.4
1,1,1620,C33,C33
2,2,E8384,V944XXA,V94.4
3,3,07052,B170,B17.0
4,4,44489,I748,I74.8
...,...,...,...,...
6979,6979,64421,O6014X0,O60.14
6980,6980,30113,F340,F34.0
6981,6981,25090,E118,E11.8
6982,6982,075,B2790,B27.90


In [11]:

# Load the original vocabulary file
vocab_file = '../vocab_stoi.csv'
original_vocab_df = pd.read_csv(vocab_file)  # ICD-9 vocabulary
original_vocab_df.rename(columns={'String': 'ICD-9'}, inplace=True)  # Rename to align with the column name in final results

# Merge the two DataFrames on 'Token Index' for comparison
merged_df = pd.merge(original_vocab_df, final_mapped_results_df, on='Token Index', suffixes=('_original', '_mapped'))

# Check for mismatches in the ICD-9 codes
mismatched_icd9 = merged_df[merged_df['ICD-9_original'] != merged_df['ICD-9_mapped']]

# Output results
if mismatched_icd9.empty:
    print("All ICD-9 codes and Token Index values match!")
    final_mapped_results_df.to_csv('vocab_stoi_icd10.csv', index=False)
else:
    print("Mismatches found:")
    print(mismatched_icd9)

if final_mapped_results_df['ICD-10-Align-Hierarchy'].isna().sum() != 0:
    print(f'There are still {final_mapped_results_df['ICD-10-Align-Hierarchy'].isna().sum()} missing ICD-10 codes.')
    print(final_mapped_results_df[final_mapped_results_df['ICD-10-Align-Hierarchy'].isna()]['ICD-9'].to_list())
else:
    print('All ICD-10 codes are successfully mapped.')

All ICD-9 codes and Token Index values match!
All ICD-10 codes are successfully mapped.


In [12]:
if final_mapped_results_df['ICD-10'].unique().shape[0] != original_vocab_df.shape[0]:
    num_unique_icd10 = final_mapped_results_df['ICD-10'].unique().shape[0]
    print(f'{original_vocab_df.shape[0]-num_unique_icd10} ICD-10 codes are duplicated.')
    assert original_vocab_df['ICD-9'].unique().shape[0] == original_vocab_df.shape[0]
    print('But orginal ICD-9 codes are unique, so the model can be trained.')
    print('We use ICD-9 codes (Toekn Index) for the code sequence and ICD-10 codes for semantic consistency to train the model.')

1027 ICD-10 codes are duplicated.
But orginal ICD-9 codes are unique, so the model can be trained.
We use ICD-9 codes (Toekn Index) for the code sequence and ICD-10 codes for semantic consistency to train the model.


### Step 3: Text Descriptions

In [13]:
import csv
import json
import time
from tqdm import tqdm

# Load hierarchical graph
with open("icd10_hierarchical_graph.json", "r") as f:
    hierarchical_graph = json.load(f)

# Load vocab_stoi_icd10.csv
csv_file = "vocab_stoi_icd10.csv"

# Function to get SHORT-TITLE from hierarchical graph
def get_short_title(icd10_code):
    node = hierarchical_graph.get(icd10_code)
    return node["description"] if node else "Description not available."

# Load the CSV as a DataFrame
df = pd.read_csv(csv_file)

# Add or update the SHORT-TITLE and LONG-TITLE columns
if "SHORT-TITLE" not in df.columns:
    df["SHORT-TITLE"] = ""
# if "LONG-TITLE" not in df.columns:
#     df["LONG-TITLE"] = ""

# Process each row with tqdm for progress tracking
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    icd10_code = row["ICD-10-Align-Hierarchy"]

    # Get short title
    short_title = get_short_title(icd10_code)
    df.at[index, "SHORT-TITLE"] = short_title


# Save the updated DataFrame back to the CSV
df.to_csv(csv_file, index=False)
print(f"Updated CSV saved to {csv_file}")


100%|██████████| 6984/6984 [00:00<00:00, 18940.39it/s]

Updated CSV saved to vocab_stoi_icd10.csv


### Step 4: Save semantic embeddings for the whole vocabulary

**Run data/mimiciii/1.4_convert_icd10/icd10_part/gen_vocab_icd10_semantic_embed.py, and then execute the following notebook**

In [14]:
import numpy as np
# Load the .npz file
file_path = 'vocab_semantic_embed.npz'
vocab_semantic_embed = np.load(file_path)

# Access the arrays stored in the .npz file
# Assuming the .npz file contains arrays with specific keys
for key in vocab_semantic_embed.files:
    print(f"{vocab_semantic_embed[key].shape} array loaded with key: {key}")
    # print(f"{key}: {data[key]}")

(6984,) array loaded with key: token_index
(6984,) array loaded with key: icd9_code
(6984,) array loaded with key: icd10_code
(6984,) array loaded with key: icd10_align
(6984, 768) array loaded with key: embedding_arr


### Step 5: Save hierarchy embeddings for the complete ICD-10 set.

**Run get_complete_icd10_hierarchy_embed.py**